# Pipeline de usuarios + análisis de distancias

En este notebook construimos un **pipeline de preprocesamiento** para un dataset de usuarios y calculamos distancias Euclídea y Manhattan entre dos usuarios después del procesamiento.

## 1. Importar librerías y crear el DataFrame

En esta celda creamos el DataFrame `df` con la información de usuarios.

In [15]:
import pandas as pd
import numpy as np

# 1) Crear DataFrame de usuarios
df = pd.DataFrame({
    'Cliente_ID': [101, 102, 103, 104, 105, 106],
    'Edad': [22, 45, 31, 28, np.nan, 40],
    'Visitas_Semana': [3, 1, 5, 2, 0, np.nan],
    'Gasto_Mensual': [12000, 250000, 80000, np.nan, 15000, 60000],
    'Ciudad': ['Santiago', 'Valparaíso', 'Santiago', 'Concepción', 'Santiago', None],
    'Plan': ['Standard', 'Premium', 'Standard', 'Standard', 'Premium', 'Premium']
})

df

,Cliente_ID,Edad,Visitas_Semana,Gasto_Mensual,Ciudad,Plan
0,101,22.0,3.0,12000.0,Santiago,Standard
1,102,45.0,1.0,250000.0,Valparaíso,Premium
2,103,31.0,5.0,80000.0,Santiago,Standard
3,104,28.0,2.0,NaN,Concepción,Standard
4,105,NaN,0.0,15000.0,Santiago,Premium
5,106,40.0,NaN,60000.0,None,Premium


## 2. Identificar columnas numéricas y categóricas

Seleccionamos las columnas que usaremos como características y las separamos en numéricas y categóricas.

In [16]:
# 2) Definir columnas numéricas y categóricas
num_cols = ['Edad', 'Visitas_Semana', 'Gasto_Mensual']
cat_cols = ['Ciudad', 'Plan']

num_cols, cat_cols

(['Edad', 'Visitas_Semana', 'Gasto_Mensual'], ['Ciudad', 'Plan'])

## 3. Definir transformaciones para cada tipo de variable

Creamos un pipeline para variables numéricas (imputación + escalado) y otro para categóricas (imputación + One-Hot Encoding).

In [17]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# 3) Pipeline para variables numéricas
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# 3) Pipeline para variables categóricas
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(missing_values=None,strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

## 4. Crear el ColumnTransformer

Unimos ambos pipelines en un `ColumnTransformer` que aplica cada transformación al grupo de columnas correspondiente.

In [18]:
from sklearn.compose import ColumnTransformer

# 4) ColumnTransformer con numéricas y categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

## 5. Crear el pipeline completo

Construimos un `Pipeline` que por ahora solo contiene el preprocesador.

In [19]:
# 5) Pipeline completo de preprocesamiento
from sklearn.pipeline import Pipeline

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor)
])

## 6. Aplicar el pipeline al dataset

Ajustamos el pipeline con los datos y obtenemos la matriz procesada `X_proc`.

In [20]:
# 6) Aplicar transformaciones al dataset
X = df[num_cols + cat_cols]
X_proc = pipeline.fit_transform(X)

X_proc

array([[-1.4248768 ,  0.52999894, -0.84213417,  0.        ,  1.        ,
         0.        ,  0.        ,  1.        ],
       [ 1.60024626, -0.74199852,  2.12716853,  0.        ,  0.        ,
         1.        ,  1.        ,  0.        ],
       [-0.241133  ,  1.8019964 ,  0.00623803,  0.        ,  1.        ,
         0.        ,  0.        ,  1.        ],
       [-0.63571427, -0.10599979, -0.2432832 ,  1.        ,  0.        ,
         0.        ,  0.        ,  1.        ],
       [-0.241133  , -1.37799724, -0.80470598,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ],
       [ 0.94261081, -0.10599979, -0.2432832 ,  0.        ,  1.        ,
         0.        ,  1.        ,  0.        ]])

## 7. Calcular distancias entre dos usuarios

Calculamos distancias Euclídea y Manhattan entre el usuario 0 (Cliente 101) y el usuario 1 (Cliente 102) usando los datos procesados.

In [21]:
from sklearn.metrics import pairwise_distances

# 7) Distancias entre el usuario 0 y 1 en el espacio procesado
dist_eucl = pairwise_distances(X_proc[[0]], X_proc[[1]], metric='euclidean')[0, 0]
dist_manh = pairwise_distances(X_proc[[0]], X_proc[[1]], metric='manhattan')[0, 0]

print('Distancia Euclídea (procesada)  101-102:', dist_eucl)
print('Distancia Manhattan (procesada) 101-102:', dist_manh)

Distancia Euclídea (procesada)  101-102: 4.856552852242064
Distancia Manhattan (procesada) 101-102: 11.26642322039482


## 8. Comentario final

Las distancias calculadas usan variables: 

- Numéricas imputadas y escaladas.
- Categóricas imputadas y codificadas con One-Hot.

Así, cada característica contribuye de forma más equilibrada a la distancia entre usuarios.

Celda A: distancias SIN escalado (solo numéricas crudas, con NaN rellenados)

In [22]:
from sklearn.impute import SimpleImputer
from sklearn.metrics import pairwise_distances
import numpy as np

# A) Distancias sin escalado (solo variables numéricas con imputación)
X_raw_num = df[num_cols].copy()

# Imputamos faltantes con la mediana, pero NO escalamos
imputer_raw = SimpleImputer(strategy='median')
X_raw_num_imp = imputer_raw.fit_transform(X_raw_num)

dist_eucl_raw = pairwise_distances(
    X_raw_num_imp[[0]], X_raw_num_imp[[1]], metric='euclidean'
)[0, 0]

dist_manh_raw = pairwise_distances(
    X_raw_num_imp[[0]], X_raw_num_imp[[1]], metric='manhattan'
)[0, 0]

print("SIN escalado (solo numéricas, imputadas)")
print("Distancia Euclídea  101-102:", dist_eucl_raw)
print("Distancia Manhattan 101-102:", dist_manh_raw)


SIN escalado (solo numéricas, imputadas)
Distancia Euclídea  101-102: 238000.0011197479
Distancia Manhattan 101-102: 238025.0


Celda B: distancias CON escalado (lo que ya hace el pipeline completo)

In [23]:
# B) Distancias CON escalado (pipeline completo: num + cat, imputado, codificado, escalado)
X_proc = pipeline.fit_transform(df[num_cols + cat_cols])

dist_eucl_proc = pairwise_distances(
    X_proc[[0]], X_proc[[1]], metric='euclidean'
)[0, 0]

dist_manh_proc = pairwise_distances(
    X_proc[[0]], X_proc[[1]], metric='manhattan'
)[0, 0]

print("CON escalado (pipeline completo)")
print("Distancia Euclídea  101-102:", dist_eucl_proc)
print("Distancia Manhattan 101-102:", dist_manh_proc)


CON escalado (pipeline completo)
Distancia Euclídea  101-102: 4.856552852242064
Distancia Manhattan 101-102: 11.26642322039482
